<a href="https://colab.research.google.com/github/DhruvalPtl/quant-kit/blob/main/quant-kit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# quant-kit — GGUF Quantization Pipeline

> **GitHub**: [DhruvalPtl/quant-kit](https://github.com/DhruvalPtl/quant-kit)  
> **HuggingFace**: [Dhptl](https://huggingface.co/Dhptl)

### Before you start:
1. `Runtime → Change runtime type → GPU (T4)`
2. Add your HF token to Colab Secrets (🔑 key icon, left sidebar):
   - Name: `HF_TOKEN` | Value: your token from https://huggingface.co/settings/tokens

Run cells **top to bottom**. If any cell fails, paste the error in our chat.


In [1]:
# ═══════════════════════════════════════════════════════════
# Cell 1 — Check runtime (GPU / Disk / RAM)
# ═══════════════════════════════════════════════════════════
import subprocess, shutil, psutil

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'[OK] GPU  : {gpu.stdout.strip()}')
else:
    print('[!!] No GPU! Go to Runtime > Change runtime type > T4 GPU')

disk = shutil.disk_usage('/')
print(f'[OK] Disk : {disk.free/1e9:.1f} GB free of {disk.total/1e9:.1f} GB')

ram = psutil.virtual_memory()
print(f'[OK] RAM  : {ram.available/1e9:.1f} GB available of {ram.total/1e9:.1f} GB')

if disk.free/1e9 < 50:
    print('[!!] Less than 50GB free — may be tight for 12B model')

[OK] GPU  : Tesla T4, 15360 MiB
[OK] Disk : 70.4 GB free of 120.9 GB
[OK] RAM  : 12.5 GB available of 13.6 GB


In [2]:
# ═══════════════════════════════════════════════════════════
# Cell 2 — Clone quant-kit & run Linux setup
# ═══════════════════════════════════════════════════════════
import os

REPO    = 'https://github.com/DhruvalPtl/quant-kit.git'
WORKDIR = '/content/quant-kit'

if os.path.exists(WORKDIR):
    print('[OK] quant-kit already cloned — pulling latest...')
    os.system(f'git -C {WORKDIR} pull')
else:
    print('[->] Cloning quant-kit...')
    os.system(f'git clone {REPO} {WORKDIR}')

os.chdir(WORKDIR)
print(f'[OK] Working directory: {os.getcwd()}')
print()

# Run Linux setup: downloads llama.cpp binaries + conversion scripts
# This handles .tar.gz extraction, symlink flattening, and .so versioned links
!python setup_linux.py

[->] Cloning quant-kit...
[OK] Working directory: /content/quant-kit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 11.8 MB/s eta 0:00:00

  quant-kit — Linux / Colab Setup
[OK] OS: Linux x86_64
[OK] Disk: 65.6 GB free of 112.6 GB total


[->] Installing Python packages...
[OK] Python packages installed

[OK] CUDA GPU detected


[->] Fetching latest llama.cpp release info...
[OK] Latest release: b9564
  Available Linux builds:
    llama-b9564-bin-ubuntu-arm64.tar.gz
    llama-b9564-bin-ubuntu-openvino-2026.0-x64.tar.gz
    llama-b9564-bin-ubuntu-rocm-7.2-x64.tar.gz
    llama-b9564-bin-ubuntu-s390x.tar.gz
    llama-b9564-bin-ubuntu-vulkan-arm64.tar.gz
    llama-b9564-bin-ubuntu-vulkan-x64.tar.gz
    llama-b9564-bin-ubuntu-x64.tar.gz

[->] Downloading: llama-b9564-bin-ubuntu-x64.tar.gz
  100% — 14.7 MB/content/quant-kit/setup_linux.py:168: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter

In [3]:
# ═══════════════════════════════════════════════════════════
# Cell 3 — Verify llama-quantize works (symlink check)
# ═══════════════════════════════════════════════════════════
import os, re, subprocess
from pathlib import Path
from collections import defaultdict

LLAMA_CPP = Path('/content/quant-kit/llama.cpp')

# Auto-fix any missing versioned .so symlinks (3-part: libXXX.so.0.0.NNNN)
three_part = re.compile(r'^(lib.+\.so)\.(\d+)\.\d+\.\d+$')
by_base = defaultdict(list)
for f in LLAMA_CPP.glob('lib*.so.*'):
    m = three_part.match(f.name)
    if m and not f.is_symlink():
        by_base[m.group(1)].append((int(m.group(2)), f.name))

fixed = 0
for base_so, versions in by_base.items():
    versions.sort(reverse=True)
    latest, major = versions[0][1], versions[0][0]
    so_major = LLAMA_CPP / f'{base_so}.{major}'
    if not so_major.exists():
        os.symlink(latest, str(so_major))
        fixed += 1

if fixed:
    print(f'[OK] Created {fixed} missing versioned .so symlinks')

# Test llama-quantize with LD_LIBRARY_PATH
env = {**os.environ, 'LD_LIBRARY_PATH': str(LLAMA_CPP)}
result = subprocess.run([str(LLAMA_CPP / 'llama-quantize'), '--help'],
                        capture_output=True, text=True, env=env)

if result.returncode == 0:
    print('[OK] llama-quantize works!')
    print('[OK] Ready to quantize. Run Cell 4.')
else:
    print(f'[ERR] llama-quantize failed (exit {result.returncode})')
    print(result.stderr[:400])
    print()
    print('Check missing libs:')
    ldd = subprocess.run(['ldd', str(LLAMA_CPP / 'llama-quantize')],
                         capture_output=True, text=True, env=env)
    for l in ldd.stdout.splitlines():
        if 'not found' in l:
            print(f'  MISSING: {l.strip()}')

[ERR] llama-quantize failed (exit 1)


Check missing libs:


In [4]:
# ═══════════════════════════════════════════════════════════
# Cell 4 — HuggingFace authentication
# ═══════════════════════════════════════════════════════════
import os
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token

    with open('/content/quant-kit/.env', 'w') as f:
        f.write(f'hf_token = "{hf_token}"\n')

    from huggingface_hub import HfApi
    user = HfApi(token=hf_token).whoami()
    print(f'[OK] Logged in as: {user["name"]}')

except Exception as e:
    print(f'[ERR] {e}')
    print('  1. Click the 🔑 key icon in the left sidebar')
    print('  2. Add secret: Name=HF_TOKEN, Value=your_token')
    print('  3. Toggle "Notebook access" ON')

[OK] Logged in as: Dhptl


In [5]:
# ═══════════════════════════════════════════════════════════
# Cell 5 — Quantize
# Edit MODEL_ID below before running!
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

MODEL_ID = 'google/gemma-4-12B'           # <-- change for different models
QUANTS   = 'Q4_K_M Q5_K_M Q8_0 IQ4_XS'  # <-- quant types to produce

# --delete-src  : frees ~24GB after FP16 conversion (critical on 120GB Colab disk)
# --keep-fp16   : add this flag if you want to keep FP16 for retry without redownload
!python quantize.py \
    --model {MODEL_ID} \
    --quants {QUANTS} \
    --delete-src


  quant-kit — GGUF Quantization Tool
  Model  : google/gemma-4-12B
  Quants : Q4_K_M, Q5_K_M, Q8_0, IQ4_XS

[OK] llama.cpp binaries found
-> Downloading google/gemma-4-12B from HuggingFace...
-> This may take a while depending on model size and internet speed
-> Free disk space: 65.5 GB
Fetching 8 files: 100% 8/8 [02:38<00:00, 19.80s/it]
Download complete: 100% 24.0G/24.0G [02:38<00:00, 156MB/s]                INFO:hf-to-gguf:Loading model: gemma-4-12B

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`
INFO:hf-to-gguf:Model architecture: Gemma4UnifiedForConditionalGeneration

You can update Transformers with the command `pip install --upgrade transformers`. If

In [6]:
# ═══════════════════════════════════════════════════════
# SAVE TO DRIVE — Run after Cell 5 finishes, before sleep
# ═══════════════════════════════════════════════════════
from google.colab import drive
import shutil, os
from pathlib import Path

# Mount your Drive
drive.mount('/content/drive')

SRC  = Path('/content/quant-kit/output/gemma-4-12B')
DEST = Path('/content/drive/MyDrive/quant-kit-output/gemma-4-12B')

DEST.mkdir(parents=True, exist_ok=True)

# Copy all GGUFs to Drive
files = list(SRC.glob('*.gguf'))
print(f'Copying {len(files)} GGUF files to Drive...')
for f in files:
    dest_file = DEST / f.name
    if not dest_file.exists():
        print(f'  Copying {f.name} ({f.stat().st_size/1e9:.1f} GB)...')
        shutil.copy2(str(f), str(dest_file))
        print(f'  Done!')
    else:
        print(f'  Already in Drive: {f.name}')

print()
print('Files saved to Drive:')
for f in sorted(DEST.glob('*.gguf')):
    print(f'  {f.name}  ({f.stat().st_size/1e9:.1f} GB)')
print()
print('Sleep well! Resume tomorrow with the Load from Drive cell.')

Mounted at /content/drive
Copying 4 GGUF files to Drive...
  Copying gemma-4-12B-Q8_0.gguf (12.7 GB)...
  Done!
  Copying gemma-4-12B-IQ4_XS.gguf (6.7 GB)...
  Done!
  Copying gemma-4-12B-Q4_K_M.gguf (7.4 GB)...
  Done!
  Copying gemma-4-12B-Q5_K_M.gguf (8.5 GB)...
  Done!

Files saved to Drive:
  gemma-4-12B-IQ4_XS.gguf  (6.7 GB)
  gemma-4-12B-Q4_K_M.gguf  (7.4 GB)
  gemma-4-12B-Q5_K_M.gguf  (8.5 GB)
  gemma-4-12B-Q8_0.gguf  (12.7 GB)

Sleep well! Resume tomorrow with the Load from Drive cell.


In [ ]:
# ═══════════════════════════════════════════════════════
# RESUME FROM DRIVE — Run tomorrow morning
# ═══════════════════════════════════════════════════════
from google.colab import drive, userdata
import shutil, os
from pathlib import Path

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Clone quant-kit (code only, no models)
os.system('git clone https://github.com/DhruvalPtl/quant-kit.git /content/quant-kit')
os.chdir('/content/quant-kit')

# 3. Restore GGUFs from Drive
SRC  = Path('/content/drive/MyDrive/quant-kit-output/gemma-4-12B')
DEST = Path('/content/quant-kit/output/gemma-4-12B')
DEST.mkdir(parents=True, exist_ok=True)

print('Restoring GGUFs from Drive...')
for f in sorted(SRC.glob('*.gguf')):
    dest_file = DEST / f.name
    if not dest_file.exists():
        print(f'  Restoring {f.name}...')
        shutil.copy2(str(f), str(dest_file))
    else:
        print(f'  Already there: {f.name}')

# 4. HF login
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
with open('.env', 'w') as f:
    f.write(f'hf_token = "{hf_token}"\n')

from huggingface_hub import HfApi
user = HfApi(token=hf_token).whoami()
print(f'\n[OK] Logged in as: {user["name"]}')

# 5. Show what's restored
print('\nFiles ready:')
for f in sorted(DEST.glob('*.gguf')):
    print(f'  {f.name}  ({f.stat().st_size/1e9:.1f} GB)')

print('\nReady! Run benchmark → model card → upload cells.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 6 — Benchmark
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

MODEL_NAME = MODEL_ID.split('/')[-1]

# ngl=99: offload all layers to GPU (T4 15GB VRAM — enough for 12B Q4/Q5)
# ngl=0:  CPU only — use if GPU VRAM not enough for the model
!python benchmark.py --model {MODEL_NAME} --ngl 99

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 7 — Generate model card (README)
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

HF_AUTHOR = 'Dhptl'   # <-- your HuggingFace username

!python model_card.py \
    --model {MODEL_NAME} \
    --original {MODEL_ID} \
    --author {HF_AUTHOR}

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 8 — Upload to HuggingFace
# Creates Dhptl/gemma-4-12B-GGUF automatically
# ═══════════════════════════════════════════════════════════
import os
os.chdir('/content/quant-kit')

!python upload.py \
    --model {MODEL_NAME} \
    --author {HF_AUTHOR}

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 9 — Cleanup (run after upload to free disk for next model)
# ═══════════════════════════════════════════════════════════
import shutil, os
from pathlib import Path

model_output = Path(f'/content/quant-kit/output/{MODEL_NAME}')
if model_output.exists():
    shutil.rmtree(str(model_output))
    print(f'[OK] Cleaned: {model_output}')

disk = shutil.disk_usage('/')
print(f'[OK] Disk after cleanup: {disk.free/1e9:.1f} GB free')
print()
print('To quantize another model:')
print('  1. Change MODEL_ID in Cell 5')
print('  2. Re-run Cells 5 → 6 → 7 → 8 → 9')